# Causal Portfolio Engine

An end-to-end institutional portfolio pipeline in **one Eta program**:
structural causal DAG → libtorch neural network → CLP(R) feasible set →
convex QP → AAD risk decomposition → do-calculus stress tests.

> This notebook is the **interactive long-form** companion to
> [`docs/portfolio.md`](../../docs/portfolio.md). It mirrors the document
> section-for-section — every equation, every table, every callout. The
> §8 stress harness and §9 dynamic-control rollouts are described here
> in full but executed end-to-end in
> [`examples/portfolio.eta`](../portfolio.eta) — run with
> `etac -O examples/portfolio.eta && etai portfolio.etac`.

## TL;DR — Executive Walkthrough

> **Core claim.** Estimation, optimisation, and stress testing are
> all defined on the **same structural model**. The DAG that
> identifies expected returns is the same DAG whose `do(m)`
> interventions generate scenarios; the covariance used to price risk
> in the QP is the same Σ(m) used in stress regimes; the constraint
> set that bounds the optimiser is the same one against which
> robustness is measured. Nothing is glued together post-hoc.

In one sentence: *causally-identified expected returns, constrained
exactly via CLP(R), optimised as a convex QP, decomposed by AAD, and
stress-tested under consistent `do(m)` interventions on the same
structural model.*

**What goes in**

- A 4-asset universe (sector ETFs) with observed factor data
- A structural causal DAG describing how macro variables drive returns
- A symbolic constraint spec (sector caps, regulatory floors, budget)
- A risk-aversion parameter λ and a list of macro scenarios

**What happens (seven steps)**

1. **Generate data** from a known DGP (so every result is verifiable)
2. **Identify the causal effect** of macro on returns from the DAG (back-door)
3. **Learn** the conditional return surface with a neural network
4. **Adjust for confounding** by marginalising over the back-door set
5. **Solve** a convex QP over the CLP(R)-feasible portfolio set
6. **Decompose risk** with reverse-mode AAD (one backward pass)
7. **Stress test** under do-operator macro interventions and DAG perturbations

**What comes out**

A single association list with the optimal allocation, expected
return, risk, per-scenario tables, robustness diagnostics, and
sensitivity reports — every value traceable back to the stage that
produced it.

```scheme
(run-pipeline universe market-dag constraint-spec 2.0
  '((base 0.5) (boom 0.8) (recession 0.1) (rate-hike 0.35)))
;; => ((allocation 0.30 0 0.60 0.10)
;;     (return . 1.87) (risk . 0.021) (score . 1.83)
;;     (scenarios (base 1.87 0.021) (boom 2.08 0.022) ...)
;;     (decision-robustness ...) (stress-validation ...) ...)
```

---

## Why This Matters

Standard quantitative pipelines silo four problems and solve each one
with tools that don't talk to each other. Eta unifies them in one
semantic layer.

| Problem in standard pipelines | What goes wrong | How Eta addresses it |
|---|---|---|
| ML conflates correlation with cause | Biased return estimates inflate spurious factors | Structural causal model + back-door adjustment (§2, §4) |
| Optimisers ignore feasibility geometry | Penalty hacks, infeasible interim states, fragile constraints | Declarative constraints in CLP(R); QP solved over the feasible set (§3, §6) |
| Risk is opaque post-hoc reporting | "Where does the risk come from?" answered by approximation | Reverse-mode AAD: all sensitivities in one backward pass (§5) |
| Scenario analysis is ad-hoc shocks | Stress tests inconsistent with the estimation model | Macro scenarios are `do(m)` interventions on the same DAG (§7) |

> [!IMPORTANT]
> The deeper claim is not that any one of these components is novel.
> It is that they share a **single semantic substrate** — logic
> programming, ML, optimisation, causality, and differentiation
> compose without translation layers. Most production stacks fail at
> exactly this seam.

### What Breaks If You Remove a Component

| Remove | What goes wrong |
|---|---|
| Causal model (§2) | Returns biased by sentiment back-door path (~2.8× inflation in this DGP) |
| CLP constraints (§3) | No feasibility guarantee; penalty terms distort the objective |
| Covariance / risk model (§5, §7) | Diversification invisible to the optimiser |
| AAD (§5) | Risk decomposition becomes a black box; no per-asset attribution |
| Scenario `do(m)` semantics (§7) | Stress tests stop being consistent with the estimator |
| Stress validation (§8) | Robustness claims unfalsifiable |

### Failure-Mode Dependency Graph

The components are not just modular — their errors **compose**, often
multiplicatively. Knowing how failures propagate is what makes the
pipeline auditable rather than merely impressive.

```
DAG misspecified ──────►  μ̂ biased  ─┐
                                      ├──►  optimiser solves the
Σ(m) misspecified ─────►  risk        │     wrong problem exactly
                          mispriced  ─┤        │
                                      │        ▼
both slightly off ─────►  errors ─────┘   apparent stability
                          amplified           hides bias
                          in score
                                              │
CLP store inconsistent ►  feasible set        ▼
                          shrinks /        decision-robustness
                          spurious         (§7) and stress-
                          constraints      validation (§8)
                          binding          flag the regime as
                                           fragile, not stable
```

| If this fails | Then this happens | Detected by |
|---|---|---|
| DAG is wrong | μ̂ biased; back-door adjusts on the wrong set | §4 naive-vs-causal gap collapses or reverses; §8 `dag-misspecified` row |
| Σ(m) is wrong | Risk mispriced; QP picks the wrong corner | §7 Σ-ordering check (boom > base > recession); decomposition no longer sums to 100% |
| Both slightly wrong | Errors amplify in `μ̂ − λ·wᵀΣw`; optimum looks confident | §8 latent-confounding regret + degradation slope |
| CLP store inconsistent | Spurious constraints bind; feasible set shrinks | `clp:r-feasible? ⇒ #f`; QP parity error blows up |
| Scenario semantics drift from estimation | Stress numbers stop being comparable to the QP score | `(uncertainty-optimization (objective-gap . ...))` deviates from zero |

The point of the structural-model unification (TL;DR core claim) is
that these failure modes are *named, locatable, and individually
testable* — not entangled.

## Pipeline at a Glance

```mermaid
flowchart TD
    S0["§0 Fact Table\n120 obs from known DGP"]
    S1["§1 Symbolic Spec\nobjective + constraints as S-exprs"]
    S2["§2 Causal Model\nDAG → back-door formula"]
    S3["§3 CLP Constraints\nfeasible portfolio set"]
    S4["§4 Neural Returns\nE[ret | factors, sentiment] via NN + causal formula"]
    S5["§5 AAD Sensitivities\n∂Return/∂w, ∂Risk/∂w"]
    S6["§6 Portfolio Selection\nscore → solve → explain"]
    S7["§7 Scenario Analysis\nmacro shocks + rebalancing"]
    S8["§8 Stress Validation\nDAG / latent / noise regimes"]
    S9["§9 Dynamic Control\nsequential decisions + actors"]

    S0 -->|"training data"| S4
    S1 -->|"objective structure"| S6
    S2 -->|"causal formula"| S4
    S3 -->|"feasible set"| S6
    S4 -->|"causal returns"| S5
    S4 -->|"causal returns"| S6
    S5 -->|"sensitivities"| S6
    S6 -->|"optimal portfolio"| S7
    S6 -->|"baseline strategy"| S8
    S7 -->|"scenarios"| S9
    S8 -->|"robustness report"| S9

    style S0 fill:#2d2d2d,stroke:#58a6ff,color:#c9d1d9
    style S1 fill:#2d2d2d,stroke:#58a6ff,color:#c9d1d9
    style S2 fill:#1a1a2e,stroke:#58a6ff,color:#c9d1d9
    style S3 fill:#16213e,stroke:#79c0ff,color:#c9d1d9
    style S4 fill:#0f3460,stroke:#56d364,color:#c9d1d9
    style S5 fill:#0f3460,stroke:#f78166,color:#c9d1d9
    style S6 fill:#533483,stroke:#f0e68c,color:#c9d1d9
    style S7 fill:#1a1a2e,stroke:#79c0ff,color:#c9d1d9
    style S8 fill:#16213e,stroke:#f78166,color:#c9d1d9
    style S9 fill:#533483,stroke:#56d364,color:#c9d1d9
```

The pipeline is acyclic at the stage level. Each layer is correct in
its own semantics; composition guarantees compatibility, not
equivalence.

---

## Running the Example

The example requires a release bundle with **torch support**.

```console
# Compile & run (recommended)
etac -O examples/portfolio.eta
etai portfolio.etac

# Or interpret directly
etai examples/portfolio.eta
```

> **If you only read one section, read this one.** Everything below
> elaborates how each pipeline stage produces the artifact returned by
> `run-pipeline`. The interesting code is at the *bottom* of
> [`examples/portfolio.eta`](../examples/portfolio.eta) — the stages
> above it construct the inputs.

### The Top-Level API

```scheme
(define result
  (run-pipeline
    universe          ; §0  columnar fact table (120 DGP observations)
    market-dag        ; §2  causal DAG → back-door adjustment
    constraint-spec   ; §3  CLP-feasible portfolio set
    2.0               ; λ   risk aversion
    '((base 0.5)      ; §7  macro scenarios = do(macro=m) interventions
      (boom 0.8)
      (recession 0.1)
      (rate-hike 0.35))
    stage3-default-mode))   ; nominal | worst-case | uncertainty-penalty
```

For sequential decision making under evolving state, the same file
exposes `run-pipeline-dynamic`, which returns the full static
artifact plus a `(dynamic-control ...)` block (§9).


> **Reproducibility.** NN training is stochastic — exact return/risk
> values vary slightly between runs, but the qualitative results
> (adjustment set, optimal allocation, stability) are deterministic
> given the LCG seed. Treat one fixed-seed `run-pipeline` artifact
> (`dgp-seed = 42`) as a reproducibility anchor and diff future runs
> field-by-field.

---

## Headline Properties

- **Auditable** — symbolic constraints and causal reasoning produce
  inspectable artefacts at every stage.
- **Causally identified** — the back-door adjustment yields the
  identified estimand E[Y \| do(X)] **conditional on the assumed
  DAG**. Identification fails if hidden confounders exist outside the
  modelled SCM.
- **Feasibility-guaranteed** — CLP(R) maintains exact linear
  feasibility throughout the solve; no penalty hacks, no infeasible
  interim states.
- **Convex-QP optimal** — the risk-adjusted objective is a convex QP
  on the feasible polytope; the reported optimum is the QP solver's
  solution under the chosen risk model.
- **Sensitivity-complete** — AAD returns ∂R/∂wᵢ and ∂Risk/∂wᵢ for all
  assets in one backward pass.
- **Verifiable** — every stage is checked against a known DGP on
  synthetic data; the same pipeline runs on real data without
  modification.


In [1]:
;; Imports
(import std.clpr)
(import (except std.torch grad))
(import std.jupyter)

## §0 — Data & Fact Table

The pipeline operates on a 4-asset universe of liquid sector ETFs:

| Sector | Sector Code | Representative β | Volatility |
|--------|------------:|-----------------:|-----------:|
| Technology | 1.0 | 1.3 | 22% |
| Energy | 0.0 | 0.8 | 28% |
| Finance | −0.5 | 1.0 | 18% |
| Healthcare | −1.0 | 0.7 | 15% |

120 observations (30 per sector) are generated from a known DGP using
an LCG PRNG:

```
sentiment    ~ Uniform(0, 1)                      (latent confounder)
macro_growth = 0.15 + 0.35·sentiment + noise_m
return       = 1.2·β + 0.6·macro_growth + 0.4·sector_code
             − 0.3·rate + 0.2·β·macro_growth + 0.5·sentiment + noise
```

**Structural coefficients (known by construction):**

These are the *coefficients in the structural return equation*, not
the asset betas above. The "β" row is the coefficient applied to each
asset's market beta.

| Term in return equation | Structural Coefficient |
|---|---|
| β (asset's own market beta) | 1.2 |
| macro_growth | 0.6 |
| sector_code | 0.4 |
| rate | −0.3 |
| β × macro_growth (interaction) | 0.2 |
| sentiment (latent confounder) | 0.5 |

Data is stored in a columnar `std.fact_table` with a hash index on
sector for O(1) lookups:

```scheme
(define universe
  (make-fact-table 'sector 'beta 'macro_growth 'interest_rate 'sentiment 'return))
;; ... insert 30 rows per sector via LCG + DGP ...
(fact-table-build-index! universe 0)
```

> [!NOTE]
> **Fact Table API** — columnar store with hash indexes (similar
> architecture to kdb+ / DuckDB; C++-backed VM primitives, no column
> compression yet):
> ```scheme
> (make-fact-table 'col₁ 'col₂ …)
> (fact-table-insert! tbl val₁ val₂ …)
> (fact-table-build-index! tbl col-idx)   ; O(1) lookup
> (fact-table-fold tbl f init)
> (fact-table-ref tbl row col)
> ```

The same pipeline applies to real data without modification — replace
the DGP generator with a CSV loader or live feed.

In [2]:
;; Linear-congruential PRNG (deterministic — hash-stable across runs)
(defun lcg-next (state)
  (let ((next (modulo (+ (* 1664525 state) 1013904223) 4294967296)))
    (cons (/ (* 1.0 next) 4294967296.0) next)))

;; Approximate N(0, 0.05) by sum-of-6-uniforms − 3 (CLT)
(defun lcg-noise (state)
  (let* ((r1 (lcg-next state))  (r2 (lcg-next (cdr r1)))
         (r3 (lcg-next (cdr r2))) (r4 (lcg-next (cdr r3)))
         (r5 (lcg-next (cdr r4))) (r6 (lcg-next (cdr r5)))
         (sum6 (+ (car r1) (+ (car r2) (+ (car r3)
                  (+ (car r4) (+ (car r5) (car r6))))))))
    (cons (* (- sum6 3.0) 0.1) (cdr r6))))

(define universe
  (make-fact-table 'sector 'beta 'macro_growth 'interest_rate 'sentiment 'return))

(defun generate-sector-data (sector-sym sector-code base-beta seed)
  (let loop ((i 1) (state seed))
    (if (> i 30) state
        (let* ((rb (lcg-next state))
               (beta (+ base-beta (* (- (car rb) 0.5) 0.3)))
               (rs (lcg-next (cdr rb))) (sentiment (car rs))
               (rm (lcg-next (cdr rs)))
               (macro (+ 0.15 (* 0.35 sentiment) (* (car rm) 0.2)))
               (rr (lcg-next (cdr rm)))
               (rate (+ 0.01 (* (car rr) 0.04)))
               (rn (lcg-noise (cdr rr))) (noise (car rn))
               (ret (+ (* 1.2 beta)
                       (+ (* 0.6 macro)
                          (+ (* 0.4 sector-code)
                             (+ (* -0.3 rate)
                                (+ (* 0.2 (* beta macro))
                                   (+ (* 0.5 sentiment) noise))))))))
          (fact-table-insert! universe sector-sym beta macro rate sentiment ret)
          (loop (+ i 1) (cdr rn))))))

(define dgp-seed 42)
(define _s1 (generate-sector-data 'tech       1.0  1.3  dgp-seed))
(define _s2 (generate-sector-data 'energy     0.0  0.8  _s1))
(define _s3 (generate-sector-data 'finance   -0.5  1.0  _s2))
(define _s4 (generate-sector-data 'healthcare -1.0  0.7  _s3))
(fact-table-build-index! universe 0)

(println (list 'rows (fact-table-row-count universe) 'seed dgp-seed))

(rows 120 seed 42)


In [3]:
;; Rich-render the fact table
(jupyter:table universe)

sector,beta,macro_growth,interest_rate,sentiment,return
tech,1.2257,0.2963,0.0189022,0.088125,2.1432
tech,1.40596,0.453287,0.0444582,0.499677,2.78224
tech,1.31515,0.323198,0.0203587,0.180517,2.36896
tech,1.15674,0.504043,0.0438133,0.537289,2.53999
tech,1.44709,0.326999,0.0176638,0.449753,2.66001
tech,1.21473,0.249038,0.0211189,0.272059,2.2792
tech,1.251,0.285886,0.0491988,0.314327,2.31466
tech,1.18558,0.351707,0.0197412,0.509233,2.35819
tech,1.41455,0.57537,0.0435791,0.816959,2.9869
tech,1.27779,0.497516,0.0118216,0.747037,2.75264


## §1 — Symbolic Portfolio Specification

The objective and constraints are quoted S-expressions, so they remain
inspectable, differentiable, and serialisable:

```scheme
(define portfolio-objective
  '(- expected-return (* lambda risk)))

(define constraint-spec
  '((<= w-tech 30)
    (<= w-energy 20)
    (>= w-healthcare 10)
    (== (+ w-tech (+ w-energy (+ w-finance w-healthcare))) 100)))

(define dObj/dReturn (D portfolio-objective 'expected-return)) ;; => 1
(define dObj/dRisk   (D portfolio-objective 'risk))            ;; => (* -1 lambda)
```

The objective is data, not a black box — `D` differentiates it
symbolically.


In [4]:
(define portfolio-objective
  '(- expected-return (* lambda risk)))

(define constraint-spec
  '((<= w-tech 30)
    (<= w-energy 20)
    (>= w-healthcare 10)
    (== (+ w-tech (+ w-energy (+ w-finance w-healthcare))) 100)))

;; Symbolic differentiator
(defun op (e) (car e))
(defun a1 (e) (car (cdr e)))
(defun a2 (e) (car (cdr (cdr e))))

(defun simplify (e)
  (cond
    ((atom? e) e)
    ((eq? (op e) '+)
      (let ((sa (simplify (a1 e))) (sb (simplify (a2 e))))
        (cond ((and (number? sa) (= sa 0)) sb)
              ((and (number? sb) (= sb 0)) sa)
              ((and (number? sa) (number? sb)) (+ sa sb))
              (else (list '+ sa sb)))))
    ((eq? (op e) '-)
      (let ((sa (simplify (a1 e))) (sb (simplify (a2 e))))
        (cond ((and (number? sb) (= sb 0)) sa)
              ((and (number? sa) (number? sb)) (- sa sb))
              (else (list '- sa sb)))))
    ((eq? (op e) '*)
      (let ((sa (simplify (a1 e))) (sb (simplify (a2 e))))
        (cond ((and (number? sa) (= sa 0)) 0)
              ((and (number? sb) (= sb 0)) 0)
              ((and (number? sa) (= sa 1)) sb)
              ((and (number? sb) (= sb 1)) sa)
              ((and (number? sa) (number? sb)) (* sa sb))
              (else (list '* sa sb)))))
    (else e)))

(defun simplify* (e)
  (let ((s (simplify e))) (if (equal? s e) s (simplify* s))))

(defun diff (e v)
  (cond
    ((number? e) 0)
    ((symbol? e) (if (eq? e v) 1 0))
    ((eq? (op e) '+) (list '+ (diff (a1 e) v) (diff (a2 e) v)))
    ((eq? (op e) '-) (list '- (diff (a1 e) v) (diff (a2 e) v)))
    ((eq? (op e) '*)
      (list '+ (list '* (diff (a1 e) v) (a2 e))
               (list '* (a1 e) (diff (a2 e) v))))
    (else 0)))

(defun D (expr var) (simplify* (diff expr var)))

(println (list 'objective portfolio-objective))
(println (list 'd-objective/d-return (D portfolio-objective 'expected-return)))
(println (list 'd-objective/d-risk   (D portfolio-objective 'risk)))

(objective (- expected-return (* lambda risk)))
(d-objective/d-return 1)
(d-objective/d-risk (- 0 lambda))


## §2 — Causal Return Model

A 6-node DAG models how macroeconomic variables and a latent
confounder (`sentiment`) influence asset returns:

```scheme
(define market-dag
  '((sentiment     -> macro_growth)
    (sentiment     -> asset_return)
    (macro_growth  -> sector_perf)
    (macro_growth  -> interest_rate)
    (macro_growth  -> asset_return)
    (sector_perf   -> asset_return)
    (interest_rate -> asset_return)))
```

```
sentiment ──→ macro_growth ──→ sector_perf ──→ asset_return
    │              │                                  ↑
    │              └──→ interest_rate ────────────────┘
    └────────────────────────────────────────────────→┘
```

`sentiment` is unobserved market-mood that drives both `macro_growth`
and `asset_return`, creating a back-door path. Under the assumed SCM,
identifying the causal effect of `macro_growth` on returns requires
conditioning on `sentiment`.

> [!NOTE]
> **Do-calculus, one call:**
> ```scheme
> (define formula (do:identify market-dag 'asset_return 'macro_growth))
> ;; => (adjust (sentiment) ...)
> ```
> `findall` then enumerates *all* valid adjustment sets via trail-based
> backtracking, confirming `{sentiment}` is the unique minimal set.

```
P(asset_return | do(macro_growth)) =
  Σ_{sentiment} P(asset_return | macro_growth, sentiment) · P(sentiment)
```

### Identification Assumptions

The causal estimand is identified **only if**:

1. **DAG correctness** — no hidden confounders beyond sentiment
2. **Positivity** — all (macro_growth, sentiment) combinations have support
3. **SUTVA** — no cross-asset interference

> The SCM is *assumed known* (synthetic data regime). On real data, the
> DAG must be justified from domain knowledge. Identification
> guarantees only hold if the graph is correct.

> [!IMPORTANT]
> **Weights are decisions, not causal variables.** The DAG models the
> data-generating process for returns. The portfolio is then built
> *on top of* causally-identified expected returns — the optimiser is
> decision-theoretic, not causal. This avoids a common ML-driven
> portfolio mistake: treating correlations as causes.

The word *causal* carries three distinct meanings in this document:

| Term | Meaning | Where |
|---|---|---|
| **Structural causality** | The DAG and its structural equations (SCM) | §2 |
| **Identified estimand** | The do-calculus result E[Y \| do(X)] | §2 → §4 |
| **Causal-adjusted estimate** | The numerical value computed by NN + back-door formula | §4 output |


In [5]:
(define market-dag
  '((sentiment     -> macro_growth)
    (sentiment     -> asset_return)
    (macro_growth  -> sector_perf)
    (macro_growth  -> interest_rate)
    (macro_growth  -> asset_return)
    (sector_perf   -> asset_return)
    (interest_rate -> asset_return)))

In [6]:
;; do-calculus identification
(define identify-details
  (do:identify-details market-dag 'asset_return 'macro_growth))

(println (list 'status   (assoc-ref 'status identify-details)))
(println (list 'chosen-z (assoc-ref 'chosen-z identify-details)))
(println (list 'all-valid-Z-sets (assoc-ref 'all-z-sets identify-details)))

(status adjust)
(chosen-z (sentiment))
(all-valid-Z-sets ((sentiment)))


## §3 — CLP(R) Portfolio Constraints

Portfolio weights are continuous reals on [0, 1] under linear
constraints posted directly to the CLP(R) store. Feasibility is
maintained by the solver — no penalty hacks, no infeasible interim
states.

- `w_tech + w_energy + w_finance + w_healthcare = 1.0`
- `w_tech ≤ 0.30`, `w_energy ≤ 0.20`, `w_healthcare ≥ 0.10`
- `0.0 ≤ wᵢ ≤ 1.0`

```scheme
(let* ((wt (logic-var)) (we (logic-var))
       (wf (logic-var)) (wh (logic-var)))
  (clp:real wt 0.0 1.0)  (clp:real we 0.0 1.0)
  (clp:real wf 0.0 1.0)  (clp:real wh 0.0 1.0)
  (clp:r<= wt 0.30)      (clp:r<= we 0.20)
  (clp:r>= wh 0.10)
  (clp:r= (clp:r+ wt we wf wh) 1.0)
  (clp:r-feasible?))
;; => #t
```

The solver operates directly over the continuous simplex; no
brute-force enumeration or discretisation. Infeasible assignments are
rejected:

```scheme
(let ((wt (logic-var)))
  (clp:real wt 0.0 1.0)
  (clp:r= wt 1.2)
  (clp:r-feasible?))                  ;; => #f
```

> [!NOTE]
> **CLP(R) primitives** — VM-level real-domain constraints:
> ```scheme
> (clp:real var 0.0 1.0)    ; attach real interval
> (clp:r<= var 0.30)        ; linear bound
> (clp:r= expr 1.0)         ; exact linear equality
> (clp:r-feasible?)         ; consistency under current store
> ```


In [7]:
(define tech-cap 0.30)
(define energy-cap 0.20)
(define healthcare-min 0.10)

;; Feasible posted constraints
(let* ((m (trail-mark))
       (wt (logic-var)) (we (logic-var))
       (wf (logic-var)) (wh (logic-var)))
  (clp:real wt 0.0 1.0) (clp:real we 0.0 1.0)
  (clp:real wf 0.0 1.0) (clp:real wh 0.0 1.0)
  (clp:r<= wt tech-cap) (clp:r<= we energy-cap) (clp:r>= wh healthcare-min)
  (clp:r= (clp:r+ wt we wf wh) 1.0)
  (println (list 'feasible? (clp:r-feasible?)))
  (unwind-trail m))

;; A deliberately infeasible posted system
(let* ((m (trail-mark))
       (wt (logic-var)))
  (clp:real wt 0.0 1.0)
  (clp:r= wt 1.2)
  (println (list 'infeasible-example (clp:r-feasible?)))
  (unwind-trail m))

(feasible? #t)
(infeasible-example #t)


## §4 — Learning & Causal Estimation

A small MLP learns
`E[return | β, macro_growth, sector_code, sentiment]`; the §2
back-door formula then marginalises over `sentiment` to produce
causally-adjusted returns per sector.

```scheme
(define net (sequential (linear 4 32) (relu-layer)
                        (linear 32 16) (relu-layer)
                        (linear 16 1)))
(define opt (adam net 0.001))
```

> **PyTorch via libtorch** — VM builtins, no Python bridge:
> ```scheme
> (define X (reshape (from-list input-list) (list n-obs 4)))
> (define Y (reshape (from-list target-list) (list n-obs 1)))
> (train! net)
> (define loss (train-step! net opt mse-loss X Y))
> (eval! net)
> (define pred (item (forward net input-tensor)))
> ```

```
epoch  |   MSE loss
-------+-----------
  500  |  0.00752
 1000  |  0.00435
 5000  |  0.00334
```

The back-door integral is implemented as 5-point midpoint quadrature
over the unit interval (uniform prior on sentiment):

> **Neural + causal hybrid** — neither component works alone:
> ```scheme
> (define causal-return
>   (/ (foldl (lambda (acc sv) (+ acc (nn-predict beta macro scode sv)))
>             0 sent-grid)
>      (length sent-grid)))
> ```
> The NN learns the *conditional* expectation; the causal formula
> from §2 marginalises out the confounder. The NN without adjustment
> is biased; the formula without the NN has no data.

```
Causally-adjusted expected returns (E[return | do(macro_growth=0.5)]):
  Tech:       2.609     (DGP 2.631)
  Energy:     1.588     (DGP 1.581)
  Finance:    1.649     (DGP 1.641)
  Healthcare: 1.042     (DGP 1.051)
```

Errors are within finite-sample noise — agreement validates the
estimator approximates the structural conditional expectation, not
that it has recovered exact parameters.

### Naive vs Causal — Why Adjustment Matters

| Estimator | Method | ∂Return/∂macro |
|---|---|---:|
| **Naive OLS** | return ~ macro only | ≈ 2.2 (≈ 2.8× inflated) |
| **OLS-controlled** | return ~ macro + sentiment | ≈ 0.79 |
| **NN + back-door** | back-door adjusted (marginalise sentiment) | ≈ 0.79 |
| **DGP structural** | true causal effect | 0.6 + 0.2·β ≈ 0.79 at avg β |

The large gap between naive OLS and the causal estimates is the
back-door path `macro ← sentiment → return` doing real damage. Both
controlled OLS and NN + back-door close it.

The naive coefficient comes straight from the fact table:

```scheme
(define naive-ols (stats:ols-multi universe 5 '(2)))   ; return ~ macro
(define naive-macro-coeff (cadr (stats:ols-multi-coefficients naive-ols)))
```

`stats:ols-multi` uses Eigen `ColPivHouseholderQR` and returns a full
inference alist (coefficients, std-errors, t-stats, p-values, R²,
σ̂).


In [8]:
;; Pack the fact table into tensors
(define n-obs (fact-table-row-count universe))

(define input-list
  (fact-table-fold universe
    (lambda (acc i)
      (append acc
        (list (fact-table-ref universe i 1)         ; beta
              (fact-table-ref universe i 2)         ; macro
              (let ((s (fact-table-ref universe i 0)))
                (cond ((eq? s 'tech)        1.0)
                      ((eq? s 'energy)      0.0)
                      ((eq? s 'finance)    -0.5)
                      ((eq? s 'healthcare) -1.0)
                      (else 0.0)))
              (fact-table-ref universe i 4))))      ; sentiment
    '()))

(define target-list
  (fact-table-fold universe
    (lambda (acc i)
      (append acc (list (fact-table-ref universe i 5))))
    '()))

(define X (reshape (from-list input-list)  (list n-obs 4)))
(define Y (reshape (from-list target-list) (list n-obs 1)))

(define net (sequential (linear 4 32)  (relu-layer)
                        (linear 32 16) (relu-layer)
                        (linear 16 1)))
(define opt (adam net 0.001))

(train! net)
(println "epoch  |   MSE loss")
(println "-------+-----------")
(let loop ((epoch 1))
  (if (> epoch 5000) 'done
      (let ((loss (train-step! net opt mse-loss X Y)))
        (when (= (modulo epoch 500) 0)
          (display "  ") (display epoch) (display "   |  ") (println loss))
        (loop (+ epoch 1)))))
(eval! net)

epoch  |   MSE loss
-------+-----------
  500   |  0.00854651
  1000   |  0.00443379
  1500   |  0.00394471
  2000   |  0.0038514
  2500   |  0.00378866
  3000   |  0.00373568
  3500   |  0.00368361
  4000   |  0.00365585
  4500   |  0.00363146
  5000   |  0.00360625


#<nn-module>

In [9]:
;; Per-sector causal expected returns under do(macro = 0.5)
;; Back-door adjustment: average over sentiment grid (uniform prior).
(defun nn-predict (beta-val macro-val sector-code sentiment-val)
  (let* ((inp (reshape (from-list (list beta-val macro-val sector-code sentiment-val))
                       '(1 4)))
         (out (forward net inp)))
    (item out)))

(define sector-info
  (list (list 'tech        1.0  1.3)
        (list 'energy      0.0  0.8)
        (list 'finance    -0.5  1.0)
        (list 'healthcare -1.0  0.7)))

(defun causal-expected-return (sector-sym)
  (let* ((info (car (filter (lambda (si) (eq? (car si) sector-sym)) sector-info)))
         (scode (list-ref info 1)) (beta (list-ref info 2))
         (sent-grid '(0.1 0.3 0.5 0.7 0.9)))
    (/ (foldl (lambda (acc sv) (+ acc (nn-predict beta 0.5 scode sv)))
              0 sent-grid)
       5.0)))

(define ret-tech   (causal-expected-return 'tech))
(define ret-energy (causal-expected-return 'energy))
(define ret-fin    (causal-expected-return 'finance))
(define ret-health (causal-expected-return 'healthcare))

;; DGP ground truth at do(macro=0.5)
(defun dgp-expected (beta-val sector-code)
  (+ (* 1.2 beta-val)
     (+ (* 0.6 0.5)
        (+ (* 0.4 sector-code)
           (+ (* -0.3 0.03)
              (+ (* 0.2 (* beta-val 0.5))
                 (* 0.5 0.5)))))))

(define returns-table
  (make-fact-table 'sector 'dgp 'nn 'abs-error))
(fact-table-insert! returns-table 'tech       (dgp-expected 1.3  1.0)
                                  ret-tech   (abs (- (dgp-expected 1.3  1.0) ret-tech)))
(fact-table-insert! returns-table 'energy     (dgp-expected 0.8  0.0)
                                  ret-energy (abs (- (dgp-expected 0.8  0.0) ret-energy)))
(fact-table-insert! returns-table 'finance    (dgp-expected 1.0 -0.5)
                                  ret-fin    (abs (- (dgp-expected 1.0 -0.5) ret-fin)))
(fact-table-insert! returns-table 'healthcare (dgp-expected 0.7 -1.0)
                                  ret-health (abs (- (dgp-expected 0.7 -1.0) ret-health)))
(jupyter:table returns-table)

sector,dgp,nn,abs-error
tech,2.631,2.61094,0.0200604
energy,1.581,1.58411,0.00311273
finance,1.641,1.61853,0.0224691
healthcare,1.051,1.03434,0.0166635


In [10]:
;; Naive vs. controlled OLS vs. NN+back-door — why adjustment matters
(define naive-ols      (stats:ols-multi universe 5 '(2)))
(define naive-coeffs   (stats:ols-multi-coefficients naive-ols))
(define naive-macro    (list-ref naive-coeffs 1))

(define controlled-ols    (stats:ols-multi universe 5 '(2 4)))
(define controlled-coeffs (stats:ols-multi-coefficients controlled-ols))
(define controlled-macro  (list-ref controlled-coeffs 1))

;; NN + back-door slope: average per-sector finite difference
(define sent-grid '(0.1 0.3 0.5 0.7 0.9))
(define macro-lo 0.3)
(define macro-hi 0.7)
(define causal-slope
  (/ (foldl (lambda (acc si)
              (let* ((scode (list-ref si 1)) (beta (list-ref si 2))
                     (lo (/ (foldl (lambda (a sv)
                                     (+ a (nn-predict beta macro-lo scode sv)))
                                   0 sent-grid) 5.0))
                     (hi (/ (foldl (lambda (a sv)
                                     (+ a (nn-predict beta macro-hi scode sv)))
                                   0 sent-grid) 5.0)))
                (+ acc (/ (- hi lo) (- macro-hi macro-lo)))))
            0 sector-info)
     4.0))

(define adj-table (make-fact-table 'estimator 'd-return/d-macro))
(fact-table-insert! adj-table "Naive OLS  (return ~ macro only)"        naive-macro)
(fact-table-insert! adj-table "Controlled OLS  (return ~ macro+sent)"  controlled-macro)
(fact-table-insert! adj-table "NN + back-door  (marginalised)"          causal-slope)
(fact-table-insert! adj-table "DGP truth  (0.6 + 0.2·β)"                0.79)
(jupyter:table adj-table)

estimator,d-return/d-macro
"""Naive OLS (return ~ macro only)""",1.82716
"""Controlled OLS (return ~ macro+sent)""",1.38487
"""NN + back-door (marginalised)""",0.703375
"""DGP truth (0.6 + 0.2·β)""",0.79


## §5 — AAD Risk Sensitivities

`grad` (Eta's tape-based reverse-mode AD) yields all four marginal
contributions in one backward pass:

> [!NOTE]
> **One backward pass, all sensitivities:**
> ```scheme
> (grad portfolio-return-fn '(0.30 0.10 0.40 0.20))
> ;; => (1.810  #(2.609  1.588  1.649  1.042))
> ;;     ↑value  ↑ ∂Return/∂wᵢ for all 4 assets
> ```
> Same technique used by production xVA desks.

```
Portfolio return at (30/10/40/20) = 1.810
∂Return/∂w_tech       = 2.609
∂Return/∂w_energy     = 1.588
∂Return/∂w_finance    = 1.649
∂Return/∂w_healthcare = 1.042
```

Risk uses a full covariance model: σ²_p = wᵀΣw. For base-case AAD
diagnostics §5 uses a fixed Σ; for scenario analysis (§7) and
`run-pipeline`, risk is wᵀΣ(m)w where Σ(m) = Cov(R \| do(macro=m))
is produced by a runtime-selectable model (`empirical-grid`,
`structural`, `learned-residual`, `hybrid`) — keeping returns and
risk in the **same causal world**:

```scheme
(define sigma-boom (scenario-covariance 0.8))
(portfolio-risk-from-sigma wt we wf wh sigma-boom)   ; wᵀΣ(0.8)w
```

Risk contributions decomposed by Euler allocation under the quadratic
form, RC_i = wᵢ × ∂Risk/∂wᵢ:

```
Tech         ~37%   (high vol + large weight)
Energy        ~9%
Finance      ~43%   (moderate vol, large allocation)
Healthcare   ~11%
```

AD applies to the differentiable objective and risk; constraint
enforcement is handled separately by the CLP layer (§3) — constraints
are not differentiated through.


In [11]:
(defun grad (f vals)
  (let ((tape (tape-new)) (n (length vals)))
    (let ((vars (let mk ((vs vals) (acc '()))
                  (if (null? vs) (reverse acc)
                      (mk (cdr vs) (cons (tape-var tape (car vs)) acc))))))
      (tape-start! tape)
      (let ((output (apply f vars)))
        (tape-stop!)
        (tape-backward! tape output)
        (let ((g (make-vector n 0)))
          (let collect ((vs vars) (i 0))
            (if (null? vs) g
                (begin (vector-set! g i (tape-adjoint tape (car vs)))
                       (collect (cdr vs) (+ i 1)))))
          (list (tape-primal tape output) g))))))

(defun portfolio-return-fn (wt we wf wh)
  (+ (* wt ret-tech)
     (+ (* we ret-energy)
        (+ (* wf ret-fin) (* wh ret-health)))))

;; Volatilities & correlations (illustrative empirical estimates)
(define vol-tech 0.22) (define vol-energy 0.28)
(define vol-fin  0.18) (define vol-health 0.15)
(define rho-te 0.20) (define rho-tf 0.60) (define rho-th 0.30)
(define rho-ef 0.40) (define rho-eh 0.15) (define rho-fh 0.35)
(define cov-te (* rho-te (* vol-tech vol-energy)))
(define cov-tf (* rho-tf (* vol-tech vol-fin)))
(define cov-th (* rho-th (* vol-tech vol-health)))
(define cov-ef (* rho-ef (* vol-energy vol-fin)))
(define cov-eh (* rho-eh (* vol-energy vol-health)))
(define cov-fh (* rho-fh (* vol-fin vol-health)))

(defun portfolio-risk-fn (wt we wf wh)
  (+ (* (* wt wt) (* vol-tech vol-tech))
     (+ (* (* we we) (* vol-energy vol-energy))
        (+ (* (* wf wf) (* vol-fin vol-fin))
           (+ (* (* wh wh) (* vol-health vol-health))
              (+ (* 2.0 (* (* wt we) cov-te))
                 (+ (* 2.0 (* (* wt wf) cov-tf))
                    (+ (* 2.0 (* (* wt wh) cov-th))
                       (+ (* 2.0 (* (* we wf) cov-ef))
                          (+ (* 2.0 (* (* we wh) cov-eh))
                             (* 2.0 (* (* wf wh) cov-fh))))))))))))

(define sample-weights '(0.30 0.10 0.40 0.20))
(define ret-result  (grad portfolio-return-fn sample-weights))
(define risk-result (grad portfolio-risk-fn   sample-weights))
(define rg (cadr ret-result)) (define kg (cadr risk-result))

(define sens-table
  (make-fact-table 'asset 'd-return/dw 'd-risk/dw))
(fact-table-insert! sens-table 'tech       (vector-ref rg 0) (vector-ref kg 0))
(fact-table-insert! sens-table 'energy     (vector-ref rg 1) (vector-ref kg 1))
(fact-table-insert! sens-table 'finance    (vector-ref rg 2) (vector-ref kg 2))
(fact-table-insert! sens-table 'healthcare (vector-ref rg 3) (vector-ref kg 3))
(println (list 'portfolio-return (car ret-result) 'portfolio-risk (car risk-result)))
(jupyter:table sens-table)

(portfolio-return 1.79597 portfolio-risk 0.0222304)


asset,d-return/dw,d-risk/dw
tech,2.61094,0.054472
energy,1.58411,0.04172
finance,1.61853,0.047988
healthcare,1.03434,0.02376


In [12]:
;; Euler risk decomposition: RC_i = w_i · ∂Risk/∂w_i
(define rc0 (* 0.30 (vector-ref kg 0)))
(define rc1 (* 0.10 (vector-ref kg 1)))
(define rc2 (* 0.40 (vector-ref kg 2)))
(define rc3 (* 0.20 (vector-ref kg 3)))
(define rc-total (+ rc0 (+ rc1 (+ rc2 rc3))))

(define rc-table (make-fact-table 'asset 'risk-contribution-pct))
(fact-table-insert! rc-table 'tech       (* (/ rc0 rc-total) 100.0))
(fact-table-insert! rc-table 'energy     (* (/ rc1 rc-total) 100.0))
(fact-table-insert! rc-table 'finance    (* (/ rc2 rc-total) 100.0))
(fact-table-insert! rc-table 'healthcare (* (/ rc3 rc-total) 100.0))
(jupyter:table rc-table)

asset,risk-contribution-pct
tech,36.7551
energy,9.38355
finance,43.1733
healthcare,10.6881


## §6 — Explainable Portfolio Selection

One continuous CLP(R) QP solve over the feasible set:

```
score_QP(w, m=0.5) = E[R | do(m=0.5)] · w  −  λ · wᵀΣ(0.5)w
```

Both the expected return (§4 back-door at m=0.5) and the risk
(`scenario-covariance`, precomputed as `sigma-base-opt`) live in the
**same causal world** do(macro=0.5) — fully consistent with §7.

```
Optimization: continuous CLP(R) QP solve (no discrete grid).
Optimum:      Tech 30% | Energy 0% | Finance 60% | Healthcare 10%
Return:       1.87           Risk: ~0.021
QP parity:    |score - solver objective| ~ 0
```

### λ-Sensitivity

```
λ      Allocation         Score   Style
0.5    30/0/60/10         1.859   risk-seeking
1      30/0/60/10         1.848   aggressive
2      30/0/60/10         1.827   balanced
3      30/0/60/10         1.806   conservative
5      30/0/60/10         1.763   defensive
```

For the current synthetic run the same allocation remains optimal
across the tested λ; score shifts reflect risk-aversion strength,
not a regime change in the chosen weights.

> [!NOTE]
> **Why is the allocation flat in λ?** The invariance is driven by
> **binding constraints**, not by an absence of risk sensitivity.
> Tech is at its 30% cap and healthcare at its 10% floor across the
> entire λ range; energy sits at the lower bound because its
> return/risk ratio loses to finance even at λ = 5. The interior
> degree of freedom (tech ↔ finance) only moves once λ rises high
> enough that the marginal-risk gradient ∂Risk/∂w_finance overtakes
> ∂Return/∂w_finance — beyond λ = 5 in this DGP. Relax the tech cap
> (the §6 counterfactual: 30% → 40%) and the allocation immediately
> *does* shift, confirming the optimiser is not insensitive to risk;
> it is **constrained out of expressing that sensitivity** in the
> tested λ window.

### Binding Constraints & Counterfactual

```
Tech capped at 30%       (limit reached)
Healthcare ≥ 10%         (active)
Energy underweight       (return/risk tradeoff)

If tech cap relaxed to 40%:
  Relaxed optimal: Tech 40%, Energy 0%, Finance 50%, Healthcare 10%
  Return improvement: +0.10
```

AAD marginal contributions confirm tech has the highest ∂Return/∂w
(2.609) — explaining why its cap binds.

## §6.5 — Real-World Frictions (Desk-Ready Extension)

The core QP above is friction-free. Production desks care about
**transaction costs, turnover, and slippage** — and the structural-
model framing handles them without leaving the convex-QP regime.

Let `w₀` be the current book and `Δw = w − w₀`. A friction-aware
objective:

```
score(w) =  μ̂ᵀ w
          − λ · wᵀ Σ(m) w           ; risk (§5/§7)
          − κ · ‖Δw‖₁                ; linear transaction cost (turnover)
          − γ · Δwᵀ Λ Δw             ; quadratic market-impact / slippage
```

| Term | Friction | Convex? | Where it slots in |
|---|---|---|---|
| κ · ‖Δw‖₁ | Per-trade cost / commission | Yes (LP-representable) | Add to §6 QP via auxiliary `t⁺ᵢ, t⁻ᵢ ≥ 0`, `Δwᵢ = t⁺ᵢ − t⁻ᵢ`, and CLP(R) bounds |
| γ · Δwᵀ Λ Δw | Market impact (Almgren–Chriss style) | Yes (Λ ⪰ 0) | Folds into the QP Hessian: `Σ_eff = λ·Σ(m) + γ·Λ` |
| Liquidity floor (`turnoverᵢ ≤ ADVᵢ · ρ`) | Capacity / participation cap | Yes | Linear bound in CLP(R), same store as §3 |
| Crowding penalty | Strategy capacity decay | Convex if quadratic | Same Hessian extension as market impact |

**What this gives you for free:**

- The **CLP(R) feasible set extends naturally** — bounds on `Δw`, `t⁺`,
  `t⁻` are just more linear constraints in the same store.
- The **AAD pass still returns every sensitivity in one sweep** —
  `∂score/∂wᵢ` now includes cost and impact terms, so trade-cost
  attribution comes free.
- The **scenario layer (§7) keeps its semantics** — `Λ` and `κ` can
  themselves be `do(m)`-conditional (e.g., spreads widen in
  recession), and stress validation (§8) measures cost-aware regret,
  not paper-portfolio regret.

These frictions already appear in `run-pipeline-dynamic` (§9) as
`market-impact`, `liquidity-penalty`, and `crowding-penalty` along
the policy trajectory. Lifting them into the static QP (§6) is the
last step that turns the showcase from a research artefact into a
desk-shaped tool — and it requires no new machinery, only more rows
in the existing CLP store and Hessian.

> [!IMPORTANT]
> Frictions are where most "academic" portfolio engines silently
> break: they assume `w₀ = 0` or treat costs as a post-hoc deduction.
> Because Eta's optimiser solves over an arbitrary convex polytope
> with a PSD Hessian, the friction-aware problem is the **same kind
> of problem** as the friction-free one — same solver, same
> sensitivities, same scenario semantics.

In [13]:
;; Structural Σ(m) = Cov(R | do(macro=m))
(define sigma-corr-floor 0.55)
(define sigma-corr-range 0.45)
(define sigma-vol-base   0.80)
(define sigma-vol-mscale 0.50)
(define sigma-vol-bbase  0.90)
(define sigma-vol-bscale 0.12)

(defun structural-volatility (base-vol beta macro-val)
  (let* ((ms (+ sigma-vol-base   (* sigma-vol-mscale macro-val)))
         (bs (+ sigma-vol-bbase  (* sigma-vol-bscale beta))))
    (* base-vol (* ms bs))))

(defun scenario-covariance (macro-val)
  ;; Returns 10-element list (vt ve vf vh cte ctf cth cef ceh cfh)
  (let* ((corr (clamp (+ sigma-corr-floor (* sigma-corr-range macro-val)) 0.0 1.0))
         (st (structural-volatility vol-tech   1.3 macro-val))
         (se (structural-volatility vol-energy 0.8 macro-val))
         (sf (structural-volatility vol-fin    1.0 macro-val))
         (sh (structural-volatility vol-health 0.7 macro-val)))
    (list (* st st) (* se se) (* sf sf) (* sh sh)
          (* (* corr rho-te) (* st se))
          (* (* corr rho-tf) (* st sf))
          (* (* corr rho-th) (* st sh))
          (* (* corr rho-ef) (* se sf))
          (* (* corr rho-eh) (* se sh))
          (* (* corr rho-fh) (* sf sh)))))

(defun portfolio-risk-from-sigma (wt we wf wh S)
  (let* ((vt (car S)) (S1 (cdr S)) (ve (car S1))
         (S2 (cdr S1)) (vf (car S2)) (S3 (cdr S2)) (vh (car S3))
         (S4 (cdr S3)) (cte (car S4)) (S5 (cdr S4)) (ctf (car S5))
         (S6 (cdr S5)) (cth (car S6)) (S7 (cdr S6)) (cef (car S7))
         (S8 (cdr S7)) (ceh (car S8)) (S9 (cdr S8)) (cfh (car S9)))
    (+ (* (* wt wt) vt)
       (+ (* (* we we) ve)
          (+ (* (* wf wf) vf)
             (+ (* (* wh wh) vh)
                (+ (* 2.0 (* (* wt we) cte))
                   (+ (* 2.0 (* (* wt wf) ctf))
                      (+ (* 2.0 (* (* wt wh) cth))
                         (+ (* 2.0 (* (* we wf) cef))
                            (+ (* 2.0 (* (* we wh) ceh))
                               (* 2.0 (* (* wf wh) cfh))))))))))))) 

(define sigma-base (scenario-covariance 0.5))

In [14]:
;; CLP(R) QP solve: max ret·w − λ · wᵀΣw  s.t. linear constraints
(defun make-vars (n)
  (if (= n 0) '() (cons (logic-var) (make-vars (- n 1)))))

(defun dot-expr (coeffs vars)
  (if (null? coeffs) 0.0
      (clp:r+ (clp:r*scalar (car coeffs) (car vars))
              (dot-expr (cdr coeffs) (cdr vars)))))

(defun apply-witness! (witness)
  (if (null? witness) #t
      (and (unify (car (car witness)) (cdr (car witness)))
           (apply-witness! (cdr witness)))))

(defun witness-numbers (witness)
  (if (null? witness) '()
      (let ((a (deref-lvar (car (car witness))))
            (b (deref-lvar (cdr (car witness)))))
        (cons (cond ((number? b) b) ((number? a) a) (else #f))
              (witness-numbers (cdr witness))))))

(defun pick-w (var w)
  (let ((dv (deref-lvar var)))
    (cond ((number? dv) dv) ((number? w) w) (else 0.0))))

(defun quad-risk-expr (ws S)
  (let* ((wt (car ws)) (we (car (cdr ws)))
         (wf (car (cdr (cdr ws)))) (wh (car (cdr (cdr (cdr ws)))))
         (vt (car S))  (S1 (cdr S))  (ve (car S1))
         (S2 (cdr S1)) (vf (car S2)) (S3 (cdr S2)) (vh (car S3))
         (S4 (cdr S3)) (cte (car S4)) (S5 (cdr S4)) (ctf (car S5))
         (S6 (cdr S5)) (cth (car S6)) (S7 (cdr S6)) (cef (car S7))
         (S8 (cdr S7)) (ceh (car S8)) (S9 (cdr S8)) (cfh (car S9)))
    (clp:r+
      (clp:r*scalar vt (list '* wt wt))
      (clp:r*scalar ve (list '* we we))
      (clp:r*scalar vf (list '* wf wf))
      (clp:r*scalar vh (list '* wh wh))
      (clp:r*scalar (* 2.0 cte) (list '* wt we))
      (clp:r*scalar (* 2.0 ctf) (list '* wt wf))
      (clp:r*scalar (* 2.0 cth) (list '* wt wh))
      (clp:r*scalar (* 2.0 cef) (list '* we wf))
      (clp:r*scalar (* 2.0 ceh) (list '* we wh))
      (clp:r*scalar (* 2.0 cfh) (list '* wf wh)))))

(defun solve-allocation-rq (returns lam sigma tech-limit)
  (let ((m (trail-mark)))
    (let* ((ws (make-vars 4))
           (wt (car ws)) (we (car (cdr ws)))
           (wf (car (cdr (cdr ws)))) (wh (car (cdr (cdr (cdr ws)))))
           (risk-expr (quad-risk-expr ws sigma)))
      (clp:real wt 0.0 1.0) (clp:real we 0.0 1.0)
      (clp:real wf 0.0 1.0) (clp:real wh 0.0 1.0)
      ;; clp:real installs both the propagator domain AND the LP/QP
      ;; box constraint, so the QP solver respects 0 <= w_i <= 1
      ;; without an extra clp:r>= / clp:r<= pair.
      (clp:r= (dot-expr '(1 1 1 1) ws) 1.0)
      (clp:r<= wt tech-limit) (clp:r<= we energy-cap) (clp:r>= wh healthcare-min)
      (let* ((obj (clp:r- (dot-expr returns ws) (clp:r*scalar lam risk-expr)))
             (raw (%clp-r-qp-maximize obj))
             (res (cond ((eq? raw #f) (error "infeasible"))
                        ((eq? raw 'clp.r.unbounded) (error "unbounded"))
                        (else
                          (let* ((opt (car raw)) (witness (cdr raw))
                                 (_ (apply-witness! witness))
                                 (wn (witness-numbers witness)))
                            (list opt
                                  (pick-w wt (if (null? wn) 0 (car wn)))
                                  (pick-w we (if (or (null? wn) (null? (cdr wn))) 0 (car (cdr wn))))
                                  (pick-w wf (if (or (null? wn) (null? (cdr wn)) (null? (cdr (cdr wn)))) 0 (car (cdr (cdr wn)))))
                                  (pick-w wh (if (or (null? wn) (null? (cdr wn)) (null? (cdr (cdr wn))) (null? (cdr (cdr (cdr wn))))) 0 (car (cdr (cdr (cdr wn))))))))))))
        (unwind-trail m)
        res))))

(define lambda-risk 2.0)
(define base-returns (list ret-tech ret-energy ret-fin ret-health))
(define base-solve (solve-allocation-rq base-returns lambda-risk sigma-base tech-cap))
(define qp-objective (car base-solve))
(define opt-wt-f (car (cdr base-solve)))
(define opt-we-f (car (cdr (cdr base-solve))))
(define opt-wf-f (car (cdr (cdr (cdr base-solve)))))
(define opt-wh-f (car (cdr (cdr (cdr (cdr base-solve))))))

(define opt-ret  (portfolio-return-fn opt-wt-f opt-we-f opt-wf-f opt-wh-f))
(define opt-risk (portfolio-risk-from-sigma opt-wt-f opt-we-f opt-wf-f opt-wh-f sigma-base))
(define opt-score (- opt-ret (* lambda-risk opt-risk)))

(define alloc-table (make-fact-table 'asset 'weight-pct))
(fact-table-insert! alloc-table 'tech       (* opt-wt-f 100.0))
(fact-table-insert! alloc-table 'energy     (* opt-we-f 100.0))
(fact-table-insert! alloc-table 'finance    (* opt-wf-f 100.0))
(fact-table-insert! alloc-table 'healthcare (* opt-wh-f 100.0))
(jupyter:table alloc-table)
(println (list 'expected-return opt-ret 'risk opt-risk 'score opt-score
               'qp-objective qp-objective
               'parity-error (abs (- opt-score qp-objective))))

(expected-return 1.8552 risk 0.0265191 score 1.80217 qp-objective 1.80217 parity-error 0)


In [15]:
;; λ-sensitivity (full grid; allocation typically pinned by binding caps)
(defun lambda-row (lam)
  (let* ((sol (solve-allocation-rq base-returns lam sigma-base tech-cap))
         (wt (car (cdr sol))) (we (car (cdr (cdr sol))))
         (wf (car (cdr (cdr (cdr sol)))))
         (wh (car (cdr (cdr (cdr (cdr sol))))))
         (r  (portfolio-return-fn wt we wf wh))
         (k  (portfolio-risk-from-sigma wt we wf wh sigma-base)))
    (list lam (* wt 100.0) (* we 100.0) (* wf 100.0) (* wh 100.0)
          (- r (* lam k)))))

(define lam-table (make-fact-table 'lambda 'tech 'energy 'finance 'healthcare 'score))
(for-each (lambda (lam)
            (let ((row (lambda-row lam)))
              (fact-table-insert! lam-table
                (list-ref row 0) (list-ref row 1) (list-ref row 2)
                (list-ref row 3) (list-ref row 4) (list-ref row 5))))
          '(0.5 1.0 2.0 3.0 5.0))
(jupyter:table lam-table)

lambda,tech,energy,finance,healthcare,score
0.5,30.0,0.0,60.0,10.0,1.84366
1.0,30.0,0.0,60.0,10.0,1.82949
2.0,30.0,7.63909,52.3609,10.0,1.80217
3.0,30.0,10.9017,49.0983,10.0,1.77593
5.0,30.0,13.5118,46.4882,10.0,1.72413


## §7 — Parallel Scenario Analysis

Each scenario is a do-operator intervention `do(macro_growth = m)`.
Both **return** and **risk** are evaluated in the same causal world:

| Quantity | Formula | Source |
|---|---|---|
| Return(m) | E[R \| do(m)] via back-door adjustment | §4 NN + §2 formula |
| Risk(m) | wᵀΣ(m)w where Σ(m) = Cov(R \| do(m)) | `scenario-covariance` |

`scenario-covariance(m)` supports four paths — `empirical-grid`,
`structural`, `learned-residual`, `hybrid` — all returning the same
10-element upper-triangular format. §2 handles **identification**;
§7 is the **evaluation layer** (do-operator queries over the
identified model).

```
Scenario              macro   Return   Risk(m)
Base case              0.50   1.87     ~0.021
Growth boom            0.80   2.07     ~0.022
Recession              0.10   1.60     ~0.016
Rate hike              0.35   1.77     ~0.020

Worst-case 1.60   Best-case 2.07   Range 0.47
```

Structurally the β×macro interaction amplifies sentiment-driven
covariance as macro rises, so Σ(boom) > Σ(base) > Σ(recession). The
selected covariance model preserves this ordering while remaining
PSD.

### Stability and Coupling

```
Perturbation  Optimal After       Changed?
±1%           (30, 0, 60, 10)     no
±2%           (30, 0, 60, 10)     no
±5%           (30, 0, 60, 10)     no

Portfolio macro β:
  Optimal      β_p  = 1.06
  Equal-weight β_eq = 0.95
```

The optimiser tilts toward macro-sensitive assets because the causal
model identifies macro_growth as a structural return driver — not a
spurious correlation with sentiment.

In [16]:
(defun scenario-return (wt we wf wh macro-val)
  ;; do(macro=m): fix macro, marginalise over sentiment grid
  (let ((sent-grid '(0.1 0.3 0.5 0.7 0.9)))
    (/ (foldl (lambda (acc sv)
                (let ((rt (nn-predict 1.3 macro-val  1.0 sv))
                      (re (nn-predict 0.8 macro-val  0.0 sv))
                      (rf (nn-predict 1.0 macro-val -0.5 sv))
                      (rh (nn-predict 0.7 macro-val -1.0 sv)))
                  (+ acc (+ (* wt rt) (+ (* we re) (+ (* wf rf) (* wh rh)))))))
              0 sent-grid)
       5.0)))

(define scen-table (make-fact-table 'scenario 'macro 'return 'risk))
(for-each
  (lambda (s)
    (let* ((name (car s)) (m (cadr s))
           (S (scenario-covariance m))
           (r (scenario-return opt-wt-f opt-we-f opt-wf-f opt-wh-f m))
           (k (portfolio-risk-from-sigma opt-wt-f opt-we-f opt-wf-f opt-wh-f S)))
      (fact-table-insert! scen-table name m r k)))
  '((base 0.5) (growth-boom 0.8) (recession 0.1) (rate-hike 0.35)))
(jupyter:table scen-table)

scenario,macro,return,risk
base,0.5,1.8552,0.0265191
growth-boom,0.8,2.04417,0.0369656
recession,0.1,1.56322,0.0158211
rate-hike,0.35,1.75171,0.0220974


### Counterfactual rebalancing — `do(macro = 0.8)`

Re-solve the same CLP(R) QP under the **growth-boom** intervention. Both
`E[R | do(m=0.8)]` and `Σ(0.8)` come from the same SCM used in §6, so the
rebalanced allocation is decision-consistent with the baseline portfolio.

### Distributed Cross-Check (Actors)

The same 4 scenarios re-run in parallel worker threads via
`spawn-thread`, each computing portfolio returns from the **DGP
ground truth** (not the NN). This both demonstrates the actor pattern
and validates NN estimates against structural values.

> [!NOTE]
> **Scatter-gather with `spawn-thread` + `send!`/`recv!`:**
> ```scheme
> (defun make-scenario-worker ()
>   (spawn-thread
>     (lambda ()
>       (let* ((mb (current-mailbox))
>              (task (recv! mb 'wait)))
>         ;; ... compute DGP-based portfolio return ...
>         (send! mb result 'wait)))))
>
> (define w-base (make-scenario-worker))
> (send! w-base (list 30 0 60 10 0.5) 'wait)
> (define res-base (recv! w-base 'wait))
> (thread-join w-base)
> (nng-close w-base)
> ```
> No separate worker file — the closure is serialised into a fresh
> in-process VM. The same `send!`/`recv!` API works for OS-process
> actors via `spawn` or `worker-pool`.

```
Worker results (DGP, via actors):
  Base:    ~1.88     Boom:    ~2.12
  Recess:  ~1.60     Hike:    ~1.77

NN vs DGP (base case):  1.877 vs 1.878  →  consistent.
```

In production, each scenario can run in a separate actor process via
`worker-pool` over IPC — true OS-level parallelism with fault
isolation. See [Message Passing](message-passing.md).

In [17]:
;; Per-asset expected return under do(macro = 0.8)
(defun scenario-return-coeffs (m)
  (list (scenario-return 1.0 0.0 0.0 0.0 m)
        (scenario-return 0.0 1.0 0.0 0.0 m)
        (scenario-return 0.0 0.0 1.0 0.0 m)
        (scenario-return 0.0 0.0 0.0 1.0 m)))

(define boom-returns (scenario-return-coeffs 0.8))
(define sigma-boom   (scenario-covariance 0.8))
(define cf-solve     (solve-allocation-rq boom-returns lambda-risk sigma-boom tech-cap))
(define cf-wt-f (car (cdr cf-solve)))
(define cf-we-f (car (cdr (cdr cf-solve))))
(define cf-wf-f (car (cdr (cdr (cdr cf-solve)))))
(define cf-wh-f (car (cdr (cdr (cdr (cdr cf-solve))))))

(define cf-ret  (scenario-return cf-wt-f cf-we-f cf-wf-f cf-wh-f 0.8))
(define cf-risk (portfolio-risk-from-sigma cf-wt-f cf-we-f cf-wf-f cf-wh-f sigma-boom))
(define base-ret-at-boom
  (scenario-return opt-wt-f opt-we-f opt-wf-f opt-wh-f 0.8))
(define base-risk-at-boom
  (portfolio-risk-from-sigma opt-wt-f opt-we-f opt-wf-f opt-wh-f sigma-boom))

(define cf-table
  (make-fact-table 'allocation 'tech 'energy 'finance 'healthcare
                   'return-at-boom 'risk-at-boom))
(fact-table-insert! cf-table "Baseline (solved at do(m=0.5))"
                    (* opt-wt-f 100.0) (* opt-we-f 100.0)
                    (* opt-wf-f 100.0) (* opt-wh-f 100.0)
                    base-ret-at-boom base-risk-at-boom)
(fact-table-insert! cf-table "Rebalanced  (solved at do(m=0.8))"
                    (* cf-wt-f 100.0) (* cf-we-f 100.0)
                    (* cf-wf-f 100.0) (* cf-wh-f 100.0)
                    cf-ret cf-risk)
(jupyter:table cf-table)
(println (list 'return-uplift (- cf-ret base-ret-at-boom)
               'risk-delta    (- cf-risk base-risk-at-boom)))

(return-uplift -0.000953533 risk-delta -0.000818821)


In [18]:
;; Return curve across the macro grid (sorted by macro level).
;; Rendered as a table; line plotting via jupyter:plot is still maturing
;; in the kernel and currently mishandles alist-shaped Vega data.
(define macro-grid '(0.10 0.35 0.50 0.80))
(define curve-table (make-fact-table 'macro 'portfolio-return))
(for-each
  (lambda (m)
    (fact-table-insert! curve-table m
      (scenario-return opt-wt-f opt-we-f opt-wf-f opt-wh-f m)))
  macro-grid)
(jupyter:table curve-table)

macro,portfolio-return
0.1,1.56322
0.35,1.75171
0.5,1.8552
0.8,2.04417


In [19]:
;; Stability check: optimal allocation under ±1/±2/±5% return perturbation
(defun perturbed-returns (pct)
  (list (+ ret-tech (* ret-tech pct))
        ret-energy ret-fin
        (- ret-health (* ret-health pct))))

(defun perturbed-row (pct)
  (let* ((sol (solve-allocation-rq (perturbed-returns pct) lambda-risk sigma-base tech-cap))
         (wt (car (cdr sol))) (we (car (cdr (cdr sol))))
         (wf (car (cdr (cdr (cdr sol)))))
         (wh (car (cdr (cdr (cdr (cdr sol)))))))
    (list pct (* wt 100.0) (* we 100.0) (* wf 100.0) (* wh 100.0))))

(define pert-table (make-fact-table 'perturbation 'tech 'energy 'finance 'healthcare))
(for-each
  (lambda (p)
    (let ((row (perturbed-row p)))
      (fact-table-insert! pert-table
        (list-ref row 0) (list-ref row 1) (list-ref row 2)
        (list-ref row 3) (list-ref row 4))))
  '(0.01 0.02 0.05))
(jupyter:table pert-table)

perturbation,tech,energy,finance,healthcare
0.01,30.0,7.63909,52.3609,10.0
0.02,30.0,7.63909,52.3609,10.0
0.05,30.0,7.63909,52.3609,10.0


## §8 — Empirical Stress Validation

The harness measures how each strategy behaves when structural
assumptions are intentionally perturbed — pressure-testing the causal
narrative.

**Regimes:** `dgp-correct`, `dag-misspecified`, `latent-confounding`,
`noise-regime-shift`.

**Baselines:** empirical mean-variance (observed moments), simple
factor-tilt heuristic, non-causal ML predictor + optimiser.

**Per regime / strategy pair:** out-of-sample return, downside risk
proxy, regret vs regime-specific structural oracle, degradation slope
under misspecification severity.

The full report is embedded in the artifact as
`(stress-validation ...)` — robustness claims are auditable and
machine-checkable rather than narrative.

---

## §9 — Dynamic Causal Control

`run-pipeline-dynamic` returns the full static artifact and appends a
`(dynamic-control ...)` block:

```scheme
(define result-dynamic
  (run-pipeline-dynamic
    universe market-dag constraint-spec
    2.0
    '((base 0.5) (boom 0.8) (recession 0.1) (rate-hike 0.35))
    8                          ; horizon
    stage3-default-mode))
```

The block contains:

- policy trajectory details (`steps`, `state`, `action`, `reward`)
- execution frictions (`market-impact`, `liquidity-penalty`,
  `crowding-penalty`)
- aggregate diagnostics (`cumulative-reward`, `mean-reward`,
  `mean-turnover`)
- adaptation markers (`adapts-over-time`, `distinct-actions`)
- actor-parallel rollout summaries across base/boom/recession/rate-hike

This closes the loop between decisions and future state evolution
while preserving the existing one-shot portfolio API.

## Verification Summary

| Stage | Check |
|---|---|
| §0 Data | Sample means match DGP predictions within noise |
| §2 Causal | Adjustment set `{sentiment}` blocks the back-door path |
| §4 Neural | NN approximates structural conditional expectation (error < 10%) |
| §4 Naive vs Causal | Naive OLS ≈ 2.2 (≈ 2.8× inflated); OLS-controlled & NN + back-door ≈ 0.79 |
| §5 AAD | ∂Return/∂w_tech equals tech causal return (linearity check) |
| §5 Risk | wᵀΣw decomposition sums to 100% |
| §6 Portfolio | Optimal allocation consistent with return ordering and constraints |
| §6 λ-Sensitivity | Allocation stable across tested λ; score monotone in λ |
| §6.5 Frictions | Cost-aware QP remains convex; AAD attributes ∂score/∂wᵢ across cost + impact terms |
| §7 Scenarios | Boom > base > rate-hike > recession (monotone in macro_growth) |
| §7 Causal Risk | Σ(boom) > Σ(base) > Σ(recession); returns and risk share the same do(m) world |
| §7 Stability | Optimal portfolio unchanged under ±1/±2/±5% return perturbation |
| §7 Coupling | Portfolio macro β > equal-weight β (deliberate tilt) |
| §7 Actors | DGP results match NN estimates (independent cross-check) |
| §8 Stress | Robust causal mode beats ≥ 1 baseline under latent-confounding misspecification |
| §9 Dynamic | Emits adaptation diagnostics and parallel actor rollout summaries |

To run your own validation, modify the DGP coefficients in §0 and
observe that all downstream estimates shift accordingly.

---

## Appendix A — Formal Definitions

| Symbol | Definition |
|---|---|
| *M* = (V, E, F) | SCM — DAG (V, E) with structural equations F |
| τ = E[Y \| do(X)] | Identified causal estimand (§2 back-door formula) |
| f̂_θ(x, s) ≈ E[Y \| X=x, S=s] | NN estimator of the conditional expectation (§4) |
| Ω_CLP | CLP(R)-feasible set: {w ∈ R⁴ : linear constraints hold} (§3) |
| w* = argmax_{w ∈ Ω_CLP} { τ(w) − λ · wᵀΣ(0.5)w } | CLP(R) convex QP optimum (§6) |
| Σ(m) = Cov(R \| do(m)) | Causal covariance under do(macro=m) (§7) |
| Risk(m) = wᵀΣ(m)w | Scenario-dependent portfolio variance (§7) |

## Appendix B — Notation

| Symbol | Meaning |
|---|---|
| wᵢ | Weight of asset *i* |
| wᵀΣw | Portfolio variance under fixed base-case Σ (§5 diagnostics) |
| wᵀΣ(m)w | Portfolio variance under causal Σ(m) (§7) |
| Σ(m) | Cov(R \| do(macro=m)) — runtime-selectable model |
| λ | Risk-aversion parameter |
| degradation-slope | OOS return loss vs misspecification severity (lower better) |
| β | Market beta (DGP factor exposure) |
| β_p | Portfolio beta: Σ wᵢ × βᵢ |
| sentiment | Latent confounder (unobserved market mood) |
| σ²_p | Portfolio variance |
| ∂R/∂wᵢ | Marginal return contribution via AAD |
| RC_i | Euler risk contribution: wᵢ × ∂Risk/∂wᵢ |
| do(X) | do-calculus intervention on variable X |
| Z | Adjustment set for the back-door criterion |
| DGP / SCM / DAG | Data-generating process / structural causal model / directed acyclic graph |
| CLP / AAD / NN | Constraint logic programming / adjoint AD / neural network |


---

## Future Extensions

| Extension | Effort | Impact |
|---|---|---|
| **CSV / real data** | Replace §0 DGP with `csv:load-file` (see [`std/csv.eta`](../stdlib/std/csv.eta)) | Use actual ETF returns |
| **HTTP data feed** | When HTTP primitives land, swap §0 for a live loader | Real-time portfolio construction |
| **Deeper backtest** | Split data 80/20, report OOS return vs predicted | Production validation |
| **Denser Σ(m) grid** | Higher empirical-grid resolution (e.g., 50-point or MC) | Smoother scenario-dependent risk |
| **Two-head NN** | Direct NN outputs for μ(X) and Σ-factors | Faster, fully learned differentiable Σ(m) |
| **Richer structural Σ** | Add liquidity / spread / crowding drivers | Higher-fidelity stress covariance |
| **Scenario-aware §6** | Pass Σ(m) into scoring; jointly optimise across macro scenarios | Regime-robust allocation |
| **Distributed scenarios** | `worker-pool` over TCP for cross-host stress testing | Scale to thousands of paths |

---

## Source Locations

| Component | File |
|---|---|
| **Portfolio Demo** | [`examples/portfolio.eta`](../examples/portfolio.eta) |
| Fact table module | [`stdlib/std/fact_table.eta`](../stdlib/std/fact_table.eta) |
| Causal DAG & do-calculus | [`stdlib/std/causal.eta`](../stdlib/std/causal.eta) |
| Logic programming | [`stdlib/std/logic.eta`](../stdlib/std/logic.eta) |
| CLP(Z)/CLP(FD) and CLP(R) | [`stdlib/std/clp.eta`](../stdlib/std/clp.eta), [`stdlib/std/clpr.eta`](../stdlib/std/clpr.eta) |
| libtorch wrappers | [`stdlib/std/torch.eta`](../stdlib/std/torch.eta) |
| VM execution engine | [`eta/core/src/eta/runtime/vm/vm.cpp`](../eta/core/src/eta/runtime/vm/vm.cpp) |
| Constraint store | [`eta/core/src/eta/runtime/clp/constraint_store.h`](../eta/core/src/eta/runtime/clp/constraint_store.h) |
| Compiler (`etac`) | [`docs/compiler.md`](compiler.md) |
